# 03 — EDA: Reservoir Levels & Hydrology

Brazil's power grid is ~60% hydroelectric. Reservoir levels (ENA — Energia Natural Afluente)
are the **dominant driver** of PLD. This notebook is Brazil-specific — there is no ENTSOE equivalent.

**Causal chain**: `rainfall → ENA → reservoir % → thermal dispatch needed → PLD`

**Sections**
1. Load reservoir and ENA bronze data
2. Reservoir level time series
3. ENA (inflow) analysis
4. ENA anomaly construction & validation
5. Reservoir vs. PLD correlation
6. Dry vs. wet season analysis
7. Crisis periods (2001, 2012-13, 2021)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

ROOT = Path("..")
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 130

SUBSYSTEMS = ["SE/CO", "S", "NE", "N"]
COLORS = {"SE/CO": "#1f77b4", "S": "#2ca02c", "NE": "#ff7f0e", "N": "#9467bd"}

## 1. Load Data

In [ ]:
con = duckdb.connect()

def load_parquet(name):
    path = ROOT / "data" / "bronze" / f"{name}.parquet"
    if not path.exists():
        print(f"File not found: {path} — run etl/run_pipeline.py first.")
        return None
    return con.execute(f"SELECT * FROM read_parquet('{path}')").fetchdf()

reservoir = load_parquet("reservoir")
pld       = load_parquet("pld")

if reservoir is not None:
    print("Reservoir columns:", reservoir.columns.tolist())
    print(reservoir.head(3))

## 2. Reservoir Level Time Series

In [ ]:
if reservoir is not None:
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

    for ax, sub in zip(axes, SUBSYSTEMS):
        data = reservoir[reservoir["subsystem"] == sub].sort_values("week_start")
        # The column name for reservoir % will be confirmed after running notebook 01
        # Adjust the column name below once you know the actual name from ONS API
        reservoir_col = [c for c in reservoir.columns if "reserv" in c.lower() or "nivel" in c.lower() or "armazen" in c.lower()]
        if reservoir_col:
            col = reservoir_col[0]
            ax.fill_between(data["week_start"], data[col], alpha=0.4, color=COLORS[sub])
            ax.plot(data["week_start"], data[col], color=COLORS[sub], linewidth=0.8)
            ax.set_ylabel(col[:20], fontsize=8)
        ax.set_title(f"{sub}", fontsize=10)
        ax.axhline(30, color="red", linestyle="--", linewidth=0.7, alpha=0.5, label="30% (critical)")

    axes[0].legend()
    plt.suptitle("Reservoir Storage Level by Subsystem", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Reservoir data not available — skipping plot")

## 3. ENA (Energia Natural Afluente) Analysis

ENA measures the water inflow to reservoirs in terms of energy potential (MWh).
It is published weekly by ONS for each subsystem.

In [ ]:
# ENA may be in the same reservoir table or a separate resource
# Check columns
if reservoir is not None:
    ena_cols = [c for c in reservoir.columns if "ena" in c.lower()]
    print("ENA-related columns:", ena_cols)

    if ena_cols:
        fig, ax = plt.subplots(figsize=(14, 5))
        for sub in SUBSYSTEMS:
            data = reservoir[reservoir["subsystem"] == sub].sort_values("week_start")
            ax.plot(data["week_start"], data[ena_cols[0]], linewidth=0.8,
                    label=sub, color=COLORS[sub])
        ax.legend()
        ax.set_title("ENA (Natural Inflow) by Subsystem")
        ax.set_ylabel("MWmed")
        plt.tight_layout()
        plt.show()

## 4. ENA Anomaly Construction & Validation

The **ENA anomaly** is the single most predictive feature:

```
ena_anomaly = ena_roll_4w / historical_mean(ena for same week-of-year)
```

- Values < 1.0 → below-average inflow → low reservoirs → high PLD
- Values > 1.0 → above-average inflow → full reservoirs → low PLD

In [ ]:
# Build ENA anomaly manually here to validate the silver.py implementation
if reservoir is not None and ena_cols:
    ena_col = ena_cols[0]

    # Work on SE/CO as example
    seco_ena = reservoir[reservoir["subsystem"] == "SE/CO"].sort_values("week_start").copy()
    seco_ena["week_start"] = pd.to_datetime(seco_ena["week_start"])
    seco_ena["week_of_year"] = seco_ena["week_start"].dt.isocalendar().week.astype(int)

    # 4-week rolling mean (lagged)
    seco_ena["ena_roll_4w"] = seco_ena[ena_col].rolling(4, min_periods=4).mean().shift(1)

    # Historical average for same week-of-year (expanding, lagged to avoid leakage)
    seco_ena["ena_hist_avg"] = (
        seco_ena.groupby("week_of_year")["ena_roll_4w"]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

    seco_ena["ena_anomaly"] = seco_ena["ena_roll_4w"] / seco_ena["ena_hist_avg"].replace(0, np.nan)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(seco_ena["week_start"], seco_ena[ena_col], linewidth=0.8, label="ENA weekly")
    axes[0].plot(seco_ena["week_start"], seco_ena["ena_hist_avg"], color="red",
                 linewidth=1.2, alpha=0.7, label="Historical avg (same week)")
    axes[0].set_title("ENA SE/CO — Actual vs. Historical Average")
    axes[0].legend()

    axes[1].plot(seco_ena["week_start"], seco_ena["ena_anomaly"], color="darkorange", linewidth=0.9)
    axes[1].axhline(1.0, color="black", linewidth=0.8, linestyle="--")
    axes[1].axhline(0.7, color="red", linewidth=0.8, linestyle=":", alpha=0.7, label="0.7 (drought warning)")
    axes[1].set_title("ENA Anomaly (ena_roll_4w / hist_avg) — Key Feature")
    axes[1].set_ylabel("Ratio")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print("ENA anomaly stats:")
    print(seco_ena["ena_anomaly"].describe())

## 5. Reservoir vs. PLD Correlation

In [ ]:
if reservoir is not None and pld is not None and ena_cols:
    # Merge PLD with ENA anomaly for SE/CO
    seco_pld = pld[pld["subsystem"] == "SE/CO"][["week_start", "pld_brl_mwh"]]
    merged = pd.merge(seco_pld, seco_ena[["week_start", "ena_anomaly"]],
                      on="week_start", how="inner").dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: ENA anomaly vs PLD
    axes[0].scatter(merged["ena_anomaly"], merged["pld_brl_mwh"], alpha=0.3, s=12)
    axes[0].set_xlabel("ENA Anomaly (ratio)")
    axes[0].set_ylabel("PLD SE/CO (R$/MWh)")
    axes[0].set_title("ENA Anomaly vs PLD — Core Causal Relationship")
    axes[0].axvline(1.0, color="red", linestyle="--", alpha=0.5)

    corr_val = merged["ena_anomaly"].corr(merged["pld_brl_mwh"])
    axes[0].text(0.05, 0.95, f"Pearson r = {corr_val:.3f}",
                transform=axes[0].transAxes, va="top",
                bbox=dict(facecolor="white", alpha=0.7))

    # Dual-axis time series
    ax2 = axes[1].twinx()
    axes[1].plot(merged["week_start"], merged["pld_brl_mwh"],
                 color="blue", linewidth=0.8, alpha=0.7, label="PLD")
    ax2.plot(merged["week_start"], merged["ena_anomaly"],
             color="green", linewidth=0.8, alpha=0.7, label="ENA anomaly")
    axes[1].set_ylabel("PLD (R$/MWh)", color="blue")
    ax2.set_ylabel("ENA Anomaly", color="green")
    axes[1].set_title("PLD vs ENA Anomaly — Time Series")

    plt.tight_layout()
    plt.show()

    print(f"\nCorrelation (ENA anomaly vs PLD SE/CO): {corr_val:.4f}")
    print("Negative correlation expected — lower inflow → higher price")

## 6. Dry vs. Wet Season Analysis

The Brazilian dry season runs approximately **May–October (weeks 18–44)**.
During this period, ENA drops, reservoirs are drawn down, and PLD typically rises.

In [ ]:
if pld is not None:
    pld_season = pld.copy()
    pld_season["week_start"] = pd.to_datetime(pld_season["week_start"])
    pld_season["week_of_year"] = pld_season["week_start"].dt.isocalendar().week.astype(int)
    pld_season["season"] = pld_season["week_of_year"].apply(
        lambda w: "Dry (May-Oct)" if 18 <= w <= 44 else "Wet (Nov-Apr)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Boxplot by season
    seasonal_pld = pld_season.groupby(["season", "subsystem"])["pld_brl_mwh"].median().reset_index()
    seasonal_pld_wide = seasonal_pld.pivot(index="season", columns="subsystem", values="pld_brl_mwh")
    seasonal_pld_wide.plot(kind="bar", ax=axes[0], color=[COLORS[s] for s in SUBSYSTEMS])
    axes[0].set_title("Median PLD: Dry vs. Wet Season")
    axes[0].set_ylabel("R$/MWh")
    axes[0].tick_params(axis="x", rotation=0)

    # Dry season premium
    for sub in SUBSYSTEMS:
        data = pld_season[pld_season["subsystem"] == sub]
        dry_med = data[data["season"] == "Dry (May-Oct)"]["pld_brl_mwh"].median()
        wet_med = data[data["season"] == "Wet (Nov-Apr)"]["pld_brl_mwh"].median()
        print(f"{sub:6s}: Dry median = {dry_med:.1f}, Wet median = {wet_med:.1f}, "
              f"Premium = {(dry_med/wet_med - 1)*100:+.1f}%")

    # Violin plot
    seco_data = pld_season[pld_season["subsystem"] == "SE/CO"]
    sns.violinplot(data=seco_data, x="season", y="pld_brl_mwh", ax=axes[1])
    axes[1].set_title("PLD SE/CO Distribution: Dry vs. Wet")
    axes[1].set_ylabel("R$/MWh")

    plt.tight_layout()
    plt.show()

## 7. Crisis Periods

Brazil has experienced 3 major hydro crises:
- **2001**: Energy rationing crisis ("apagão") — severe drought
- **2012-2013**: Reservoir drawdown, PLD at ceiling for extended periods
- **2021**: Worst drought in 91 years — PLD hit R$559/MWh ceiling

In [ ]:
if pld is not None:
    crisis_periods = [
        ("2001-01-01", "2002-12-31", "2001 Apagão", "red"),
        ("2012-01-01", "2013-12-31", "2012-13 drought", "orange"),
        ("2021-01-01", "2021-12-31", "2021 drought", "darkred"),
    ]

    seco = pld[pld["subsystem"] == "SE/CO"].sort_values("week_start")
    seco["week_start"] = pd.to_datetime(seco["week_start"])

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(seco["week_start"], seco["pld_brl_mwh"], linewidth=0.8, color="steelblue")

    for start, end, label, color in crisis_periods:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.2, color=color, label=label)

    ax.legend()
    ax.set_title("PLD SE/CO — Brazilian Hydro Crisis Periods")
    ax.set_ylabel("R$/MWh")
    plt.tight_layout()
    plt.show()

    # Crisis vs. normal period stats
    for start, end, label, _ in crisis_periods:
        mask = (seco["week_start"] >= pd.Timestamp(start)) & (seco["week_start"] <= pd.Timestamp(end))
        crisis_mean = seco[mask]["pld_brl_mwh"].mean()
        normal_mean = seco[~mask]["pld_brl_mwh"].mean()
        print(f"{label}: mean PLD = {crisis_mean:.1f} vs. {normal_mean:.1f} non-crisis ({crisis_mean/normal_mean:.1f}x)")